## PDFs
- https://www.inside-sephora.com/sites/default/files/2025-06/SEPHORA_Global%20Impact%20Report%20-%20accessible.pdf
- https://www.researchgate.net/publication/373896628_Analysis_of_Sephora's_Market/link/652afbe40e4a1710e50977a0/download?_tp=eyJjb250ZXh0Ijp7ImZpcnN0UGFnZSI6InB1YmxpY2F0aW9uIiwicGFnZSI6InB1YmxpY2F0aW9uIn19

In [0]:
import os
from pathlib import Path

path = Path("/Volumes/workspace/sephora_products_and_skincare_reviews/data/pdf")
path_parsed_images = Path("/Volumes/workspace/sephora_products_and_skincare_reviews/data/pdf/parsed_images")

os.makedirs(path_parsed_images, exist_ok=True)

file_analyzed = "Sephora.pdf"

In [0]:
import pyspark.sql.functions as sf

In [0]:
df_docs = spark.read.format("binaryFile").load(str(path / file_analyzed))

In [0]:
df_parsed = df_docs.withColumn(
    "parsed_content",
    sf.expr(f"""
        ai_parse_document(
            content,
            map(
                "version", "2.0",
                "imageOutputPath", "{path_parsed_images}"
            )
        )     
    """)
)

df_parsed = df_parsed.drop("content")

In [0]:
display(df_parsed)

In [0]:
df_metadata = df_parsed.select(
    "path",
    sf.expr("parsed_content:document:pages"),
    sf.expr("parsed_content:document:elements"),
    sf.expr("parsed_content:error_status"),
    sf.expr("parsed_content:corrupted_data"),
    sf.expr("parsed_content:metadata"),
)

display(df_metadata)

In [0]:
table = "workspace.sephora_products_and_skincare_reviews.bronze_pdf_parsed"
df_parsed.write.format("delta").mode("overwrite").saveAsTable(table)